In [13]:
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import randint


from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("department_prediction_dataset.csv", keep_default_na=False)

df.head()

,patient_id,age,gender,symptoms,temperature,duration_days,blood_pressure_systolic,blood_pressure_diastolic,heart_rate,oxygen_saturation,pain_level,symptom_severity,chronic_condition,department
0,100001,10,Female,"constipation, abdominal pain",100.1,8,125,80,62,98,5,Moderate,None,Gastroenterology
1,100002,40,Female,"mood swings, stress",98.1,3,107,63,68,100,1,Mild,None,Psychiatry
2,100003,78,Female,"ear pain, hearing loss",99.5,4,126,83,66,98,4,Moderate,Diabetes,ENT
3,100004,40,Male,"rapid heartbeat, chest pain",99.3,10,138,96,83,90,7,Moderate,Hypertension,Cardiology
4,100005,45,Other,"diarrhea, stomach pain",100.2,7,107,67,67,99,5,Moderate,None,Gastroenterology


In [14]:
X = df.drop(columns=["patient_id", "department"])
y = df["department"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


text_features = ["symptoms"]

categorical_features = [
    "gender",
    "symptom_severity",
    "chronic_condition"
]

numeric_features = [
    "age",
    "temperature",
    "duration_days",
    "blood_pressure_systolic",
    "blood_pressure_diastolic",
    "heart_rate",
    "oxygen_saturation",
    "pain_level"
]




preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "symptoms"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)




# RandomizedSearchCV


In [15]:
# Decision Tree Pipeline
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Hyperparameter distributions
param_dist = {
    "classifier__max_depth": [5, 10, 15, 20, None],
    "classifier__min_samples_split": randint(2, 11),   # 2 to 10
    "classifier__min_samples_leaf": randint(1, 5)      # 1 to 4
}

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=dt_pipeline,
    param_distributions=param_dist,
    n_iter=10,                 # Try only 10 random combinations
    scoring="accuracy",
    cv=3,                      # 3-fold CV for faster training
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# Train
random_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(random_search.best_score_)

# Best model
best_dt = random_search.best_estimator_

# Prediction
y_pred = best_dt.predict(X_test)

# Evaluation
print("\nTest Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameters:
{'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 8}

Best Cross Validation Accuracy:
1.0

Test Accuracy:
1.0

Classification Report:
                  precision    recall  f1-score   support

      Cardiology       1.00      1.00      1.00      2020
       Dentistry       1.00      1.00      1.00      2012
     Dermatology       1.00      1.00      1.00      1990
             ENT       1.00      1.00      1.00      2001
   Endocrinology       1.00      1.00      1.00      1995
Gastroenterology       1.00      1.00      1.00      2017
General Medicine       1.00      1.00      1.00      2010
      Gynecology       1.00      1.00      1.00      1995
       Neurology       1.00      1.00      1.00      2009
   Ophthalmology       1.00      1.00      1.00      1986
     Orthopedics       1.00      1.00      1.00      1981
      Pediatrics       1.00      1.00    

In [18]:
# Save model
joblib.dump(best_dt, "department_prediction_model.pkl")

print("\nModel saved successfully!")


Model saved successfully!
